### Model Serving and Deployment`data-pipelines-and-scaling.ipynb` covers extract/transform/load and training. This picks up after that: you have a trained model artifact, how does it actually serve predictions to something else, an app, a batch job, another service. Covers batch vs. real-time inference, serialization, wrapping a model as an API, containerizing it, and what changes once it's live (versioning, monitoring).

#### 0. Batch vs. real-time inferenceBatch: run the model over a large set of rows on a schedule (nightly, hourly), write results somewhere (a table, a file), nothing waits on it in real time. Real-time (online): a request comes in, the model scores it, a response goes back, all within the caller's own request, latency is now a hard constraint, not just a nice-to-have.Worked comparison, fraud scoring: batch fits "re-score every account overnight, flag anything above a threshold for tomorrow's review queue", cheap, simple, but a fraudulent transaction at 2pm doesn't get caught until the next batch run. Real-time fits "score this transaction before it's authorized", catches it immediately, but now the model has to respond in milliseconds, under load, with an uptime guarantee, a much higher engineering bar for the exact same model artifact. Same model, completely different deployment shape, chosen by the LATENCY requirement of the decision it feeds, not by anything about the model itself.

#### 1. Serialization: saving a trained model as a portable artifactDifferent formats for different frameworks, all solving the same problem, getting a trained model's parameters out of the training process and into something a separate serving process can load without retraining.- `pickle`/`joblib`: standard for sklearn/XGBoost, `joblib` preferred for anything with large numpy arrays (more efficient than raw pickle for that case). Tied to the exact Python/library versions used to train, a real operational risk, loading a joblib file with a different sklearn version than it was saved with can silently break or refuse to load.- PyTorch: `state_dict` (the model's learned weights only, not the class definition), covered in `pytorch-intro.ipynb`'s "Saving and loading models" section, requires the model class definition to exist wherever it's loaded.- ONNX: a framework-agnostic format, export a PyTorch or sklearn model to ONNX once, then run it in any ONNX-compatible runtime (C++, Java, a mobile device) without needing Python or the original framework installed at serving time, trades some flexibility for portability and often faster inference (the ONNX runtime is optimized specifically for inference, not training).

In [ ]:
import joblibfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.datasets import make_classificationX, y = make_classification(n_samples=200, n_features=4, random_state=42)model = RandomForestClassifier(random_state=42).fit(X, y)joblib.dump(model, "fraud_model.joblib")loaded_model = joblib.load("fraud_model.joblib")print("original vs loaded predictions match:", (model.predict(X) == loaded_model.predict(X)).all())

#### 2. Wrapping a model as a real-time API (FastAPI)FastAPI pairs naturally with `pydantic-basics.ipynb`, the request/response schemas ARE pydantic models, validation (bad request shape, wrong types, out-of-range values) happens automatically before your prediction code ever runs, the same validation mechanism from that notebook, now sitting at the boundary of a live service instead of validating a dict in a script.Load the model ONCE at startup (module-level, not inside the endpoint function), loading a joblib/ONNX file from disk on every request would add real latency and is pure waste, the model doesn't change between requests.

In [ ]:
# app.py — illustrative, not executed here (fastapi/uvicorn not installed in this env)# run with: uvicorn app:app --reload# from fastapi import FastAPI# from pydantic import BaseModel, Field# import joblib# import numpy as np## app = FastAPI()# model = joblib.load("fraud_model.joblib")  # loaded once at startup, not per-request## class TransactionRequest(BaseModel):#     amount: float = Field(..., ge=0)#     narrative_length: int = Field(..., ge=0)#     mentions_irs: bool#     hour_of_day: int = Field(..., ge=0, le=23)## class FraudScoreResponse(BaseModel):#     fraud_probability: float#     flagged: bool## @app.post("/predict", response_model=FraudScoreResponse)# def predict(txn: TransactionRequest):#     features = np.array([[txn.amount, txn.narrative_length, int(txn.mentions_irs), txn.hour_of_day]])#     prob = model.predict_proba(features)[0, 1]#     return FraudScoreResponse(fraud_probability=float(prob), flagged=prob > 0.5)print("see commented app.py above — a malformed request (e.g. hour_of_day=99) gets rejected")print("by pydantic validation automatically, the endpoint function never even runs")

#### 3. Containerizing with DockerThe problem it solves: "works on my machine" — the serving environment needs the exact same Python version, library versions, and OS-level dependencies the model was trained/tested against, a container packages all of that into one portable image instead of hoping the deployment target happens to match.Worked minimal Dockerfile for the FastAPI app above:```dockerfileFROM python:3.12-slimWORKDIR /appCOPY requirements.txt .RUN pip install --no-cache-dir -r requirements.txtCOPY app.py fraud_model.joblib ./EXPOSE 8000CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]```Line by line: `FROM` picks a minimal base image (smaller image, smaller attack surface, faster pulls than a full OS image). `WORKDIR` sets the working directory inside the container. Dependencies are installed in their OWN layer, before copying application code, so Docker's layer cache can skip the (slow) `pip install` step on rebuilds where only `app.py` changed, not `requirements.txt`, a real build-speed detail, not just style. `EXPOSE` documents the port (doesn't actually publish it, that's done with `docker run -p`). `CMD` is what runs when the container starts.

#### 4. Model registry and versioningOnce a model is deployed, "which exact model artifact is live" becomes a real question, tied to `drift-monitoring.ipynb`'s retraining triggers, when drift fires and a new model gets trained, something needs to track: which training data version produced it, what its validation metrics were, whether it's currently in production or staging, and how to roll back to the previous version if the new one underperforms live.A model registry (MLflow's Model Registry is the common open-source one) stores exactly this: each trained model gets a version number, metadata (metrics, training data snapshot, timestamp), and a stage label (`staging`/`production`/`archived`). Promoting a model from staging to production becomes a metadata change, not a redeploy, and rollback is just re-pointing production at the previous version number, the artifact never disappeared.

#### 5. Scaling and monitoring a live modelScaling: run multiple copies of the serving process behind a load balancer (horizontal scaling, same idea as scaling a normal web service), stateless by design, each request is independent, the model itself doesn't need to know about other replicas. For GPU-bound models specifically, request batching (grouping several incoming requests into one forward pass) trades a small amount of added latency per request for much higher total throughput, since a GPU forward pass over a batch of 8 isn't much slower than over 1.Monitoring: two different things people mean by "monitoring" a model. Infra-level (latency, error rate, uptime, ordinary service monitoring). Model-level (is it still ACCURATE, `drift-monitoring.ipynb`'s PSI/KS-test territory, checking whether the live input distribution or the relationship between features and outcome has shifted since training). A model can be perfectly healthy on every infra metric, 100% uptime, fast responses, while being silently wrong on every prediction, infra monitoring alone can't see that, only the drift/label-quality checks can.Canary/shadow deployment: roll a new model out to a small slice of traffic (canary) or run it alongside the current production model without actually using its predictions yet (shadow), comparing its outputs to the incumbent's before trusting it with 100% of traffic, the safe way to validate "does this actually work in production" without betting the whole system on it.

#### 6. Batch vs. real-time, summarized| | Batch | Real-time ||---|---|---|| Latency requirement | none, runs on a schedule | milliseconds to seconds || Infra complexity | low, a scheduled job | higher, needs uptime/scaling/load balancing || Freshness | stale between runs (hours) | immediate || Typical fit | overnight fraud re-scoring, reporting | transaction authorization, live recommendations || Failure mode | a late/failed job delays everyone | a slow/down service blocks the caller directly |